In [1]:
from pathlib import Path
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt


#loading initial imputs and outputs 
X= np.load('../data/initial_data/function_5/initial_inputs.npy')
y = np.load('../data/initial_data/function_5/initial_outputs.npy')

#checking shape of data 
print('Inputs Shape:', X.shape)
print('Outputs Shape:', y.shape)

#creating a dataframe 
df=pd.DataFrame(X, columns=['x1', 'x2', 'x3', 'x4'])
df['y']=y

#checking DataFrame
df.head(50)

Inputs Shape: (20, 4)
Outputs Shape: (20,)


,x1,x2,x3,x4,y
0,0.191447,0.038193,0.607418,0.414584,64.443440
1,0.758653,0.536518,0.656000,0.360342,18.301380
2,0.438350,0.804340,0.210245,0.151295,0.112940
3,0.706051,0.534192,0.264243,0.482088,4.210898
4,0.836478,0.193610,0.663893,0.785649,258.370525
5,0.683432,0.118663,0.829046,0.567577,78.434389
6,0.553621,0.667350,0.323806,0.814870,57.571537
7,0.352356,0.322242,0.116979,0.473113,109.571876
8,0.153786,0.729382,0.422598,0.443074,8.847992
9,0.463442,0.630025,0.107906,0.957644,233.223610


In [3]:
# --- add new observation for Function 3 ---
new_row = pd.DataFrame([{
    'x1': 0.304,
    'x2': 0.957333,
    'x3': 0.99,
    'x4': 0.99,
    'y': 3681.946779
}, {'x1': 0.924667, 'x2': 0.99, 'x3': 0.99, 'x4': 0.99, 'y': 6886.719291}, {'x1': 0.99, 'x2': 0.99, 'x3': 0.99, 'x4': 0.99, 'y': 7915.738969}, 
                        {
        'x1': 0.99,
        'x2': 0.99,
        'x3': 0.075333,
        'x4': 0.99,
        'y': 4040.12375
    }, {
        'x1': 0.98,
        'x2': 0.99,
        'x3': 0.99,
        'x4': 0.99,
        'y': 7739.627613
    }, {
        'x1': 0.99,
        'x2': 0.98,
        'x3': 0.99,
        'x4': 0.99,
        'y': 7739.627613
    }
])

# append to df
df = pd.concat([df, new_row], ignore_index=True)

# rebuild X and y from the *updated* df
X = df[['x1', 'x2', 'x3', 'x4']].to_numpy()
y = df['y'].to_numpy()

# sanity check
print("len(df):", len(df))
print("Inputs Shape:", X.shape)
print("Outputs Shape:", y.shape)
print(df.tail())

# Check
#checking DataFrame
df.head(50)


len(df): 26
Inputs Shape: (26, 4)
Outputs Shape: (26,)
          x1    x2        x3    x4            y
21  0.924667  0.99  0.990000  0.99  6886.719291
22  0.990000  0.99  0.990000  0.99  7915.738969
23  0.990000  0.99  0.075333  0.99  4040.123750
24  0.980000  0.99  0.990000  0.99  7739.627613
25  0.990000  0.98  0.990000  0.99  7739.627613


,x1,x2,x3,x4,y
0,0.191447,0.038193,0.607418,0.414584,64.443440
1,0.758653,0.536518,0.656000,0.360342,18.301380
2,0.438350,0.804340,0.210245,0.151295,0.112940
3,0.706051,0.534192,0.264243,0.482088,4.210898
4,0.836478,0.193610,0.663893,0.785649,258.370525
5,0.683432,0.118663,0.829046,0.567577,78.434389
6,0.553621,0.667350,0.323806,0.814870,57.571537
7,0.352356,0.322242,0.116979,0.473113,109.571876
8,0.153786,0.729382,0.422598,0.443074,8.847992
9,0.463442,0.630025,0.107906,0.957644,233.223610


In [5]:
#setting parameters and making  grid 
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C


#hyperparamters for GP surrogate model and acqusition function 
rbf_lengthscale = 0.5
noise_assumption = 0.01
xi = 0.05

# grid
g = np.linspace(0.01, 0.99, 31)
x1g, x2g, x3g, x4g = np.meshgrid(g, g, g, g, indexing='ij')
X_grid = np.column_stack([x1g.ravel(), x2g.ravel(), x3g.ravel(), x4g.ravel()])


print(X_grid.shape)


(923521, 4)


In [6]:
#fit GP to current data 
kernel = (C(1.0, (1e-2, 1e2)) * RBF(length_scale=[0.5, 0.5, 0.5, 0.5], length_scale_bounds=(0.1, 3))
    + WhiteKernel(noise_level=0.01 , noise_level_bounds=(0.01, 0.05)))
gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=20, random_state=0)
gp.fit(X, y)

/opt/anaconda3/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


GaussianProcessRegressor(kernel=1**2 * RBF(length_scale=[0.5, 0.5, 0.5, 0.5]) + WhiteKernel(noise_level=0.01),
                         n_restarts_optimizer=20, normalize_y=True,
                         random_state=0)

In [7]:
from scipy.stats import norm
import numpy as np

def expected_improvement(mu, sigma, y_best, xi=0.0):
    sigma = np.maximum(sigma, 1e-12)
    imp = mu - y_best - xi
    Z = imp / sigma
    return imp * norm.cdf(Z) + sigma * norm.pdf(Z)

# predict GP on candidate set
mu, std = gp.predict(X_grid, return_std=True)

# best observed so far (maximization)
y_best = float(np.max(y))

# compute EI and select next point
ei = expected_improvement(mu, std, y_best, xi=xi)  # use your xi
ix = int(np.argmax(ei))
x_next = X_grid[ix]
print(f"x_next = ({x_next[0]:.6f}, {x_next[1]:.6f},{x_next[2]:.6f},{x_next[3]:.6f}) ")


x_next = (0.990000, 0.990000,0.990000,0.990000) 


In [6]:
#x_next check 
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import euclidean_distances

model        = gp
X_candidates = np.array(X_grid)   # shape (n_candidates, n_dims)
acq_values   = np.array(ei)   # shape (n_candidates,)
x_next       = np.array(x_next)   # shape (n_dims,)
y_best       = y_best
X_sampled    = np.array(X)   # all evaluated X so far


def _percentile_of_score(values, score):
    """Simple percentile rank: percentage of values <= score."""
    values = np.asarray(values).ravel()
    return float(100.0 * (np.sum(values <= score) / max(len(values), 1)))

def _first_length_scale_from_kernel(kernel):
    """Extract effective GP length-scale (handles RBF, Matern, etc.)."""
    if hasattr(kernel, "length_scale"):
        ls = getattr(kernel, "length_scale")
        try:
            ls = float(np.mean(ls))
        except Exception:
            ls = float(ls)
        return ls
    for attr in ("k1", "k2"):
        if hasattr(kernel, attr):
            ls = _first_length_scale_from_kernel(getattr(kernel, attr))
            if ls is not None:
                return ls
    return None

def _nearest_dist_over_ell(x_next, X_sampled, ell):
    """Compute distance from x_next to nearest sampled point ÷ ell."""
    x_next = np.asarray(x_next, dtype=float).reshape(1, -1)
    X_sampled = np.asarray(X_sampled, dtype=float)
    if X_sampled.size == 0:
        return np.nan
    dists = euclidean_distances(x_next, X_sampled).ravel()
    d_min = float(np.min(dists))
    if (ell is None) or (ell <= 0):
        return np.nan
    return d_min / ell

def _classify_strategy(sig_pct):
    """Assign strategy label from σ percentile (bounded exploration)."""
    if sig_pct <= 40:
        return "Exploit"
    elif sig_pct <= 75:
        return "Balanced"
    elif sig_pct <= 85:
        return "Explore"
    else:
        return "Too exploratory"

def sanity_check_table(model, X_candidates, acq_values, x_next, y_best, X_sampled):
    """
    Builds a 1-row DataFrame with:
      μ_raw (raw), σ_pct (scaled %), dist_over_ℓ_raw (raw), acq_pct (scaled %), Strategy
    """

    # Ensure arrays are the correct shape
    X_candidates = np.asarray(X_candidates, dtype=float)
    acq_values = np.asarray(acq_values, dtype=float).ravel()
    x_next = np.asarray(x_next, dtype=float).ravel()

    # 1) Predict μ, σ for all candidates (for percentiles) and for x_next
    mu_all, sigma_all = model.predict(X_candidates, return_std=True)
    mu_xn, sigma_xn = model.predict(x_next.reshape(1, -1), return_std=True)
    mu_xn = float(mu_xn.ravel()[0])
    sigma_xn = float(sigma_xn.ravel()[0])

    # 2) Percentiles for σ and acquisition
    sig_pct = _percentile_of_score(sigma_all, sigma_xn)

    # Find acquisition value at x_next by matching to nearest candidate
    idx_closest = np.argmin(np.sum((X_candidates - x_next.reshape(1, -1))**2, axis=1))
    acq_xn = float(acq_values[idx_closest])
    acq_pct = _percentile_of_score(acq_values, acq_xn)

    # 3) Distance / ell
    ell = None
    if hasattr(model, "kernel_"):
        ell = _first_length_scale_from_kernel(model.kernel_)
    dist_over_ell = _nearest_dist_over_ell(x_next, X_sampled, ell)

    # 4) Strategy classification
    strategy = _classify_strategy(sig_pct)

    # 5) Build the 1-row DataFrame (rounded for readability)
    data = {
        "μ_raw (raw)":            np.round(mu_xn, 3),
        "σ_pct (scaled %)":       int(np.round(sig_pct)),
        "dist_over_ℓ_raw (raw)":  np.round(dist_over_ell, 3) if np.isfinite(dist_over_ell) else np.nan,
        "acq_pct (scaled %)":     int(np.round(acq_pct)),
        "Strategy":               strategy
    }

    return pd.DataFrame([data])
   

result_df = sanity_check_table(model, X_candidates, acq_values, x_next, y_best, X_sampled)
result_df

,μ_raw (raw),σ_pct (scaled %),dist_over_ℓ_raw (raw),acq_pct (scaled %),Strategy
0,7473.673,2,0.0,100,Exploit
